# 03 — Align Images with TweakReg

**Purpose:** Run TweakReg to compute a refined astrometric solution for each quasar's
exposures and save the alignment-corrected FLT files to `data/processed/<quasar>/aligned/`.

**Inputs:** FLT files from `data/raw/<quasar>/`

**Outputs:** Alignment-corrected FLT copies in `data/processed/<quasar>/aligned/`

**After running:** Use `/check-alignment` to review TweakReg logs before proceeding.

**Next step:** `04_drizzle.ipynb`

## Imports

In [ ]:
import os
import shutil
import glob
from pathlib import Path
from drizzlepac import tweakreg

## Define paths

In [ ]:
RAW_DIR = Path('../data/raw')
PROCESSED_DIR = Path('../data/processed')
quasar_dirs = sorted([d for d in RAW_DIR.iterdir() if d.is_dir()])
print(f'Quasars to align: {[d.name for d in quasar_dirs]}')

## Copy raw FLT files to aligned/ directories

TweakReg updates WCS headers in-place, so we work on copies to preserve the originals.

In [ ]:
for qdir in quasar_dirs:
    aligned_dir = PROCESSED_DIR / qdir.name / 'aligned'
    aligned_dir.mkdir(parents=True, exist_ok=True)
    for flt in qdir.glob('*flt.fits'):
        dest = aligned_dir / flt.name
        if not dest.exists():
            shutil.copy2(flt, dest)
    print(f'{qdir.name}: {len(list(aligned_dir.glob("*flt.fits")))} FLT files copied to aligned/')

## Run TweakReg per quasar

In [ ]:
for qdir in quasar_dirs:
    aligned_dir = PROCESSED_DIR / qdir.name / 'aligned'
    flt_files = sorted(aligned_dir.glob('*flt.fits'))
    
    if not flt_files:
        print(f'{qdir.name}: no FLT files in aligned/ — skipping')
        continue
    
    print(f'\nAligning {qdir.name} ({len(flt_files)} exposures)...')
    
    tweakreg.TweakReg(
        files=[str(f) for f in flt_files],
        imagefindcfg={'threshold': 200, 'conv_width': 3.5},
        shiftfile=False,
        updatehdr=True,
        logfile=str(aligned_dir / 'tweakreg.log')
    )

## Next steps

Run `/check-alignment` to review TweakReg logs for each quasar.
If alignment looks good for all 6 quasars, proceed to `04_drizzle.ipynb`.